# 🎮 宝可梦对战游戏

命令行宝可梦对战游戏，玩家和电脑各选3只宝可梦进行对战。

## 宝可梦

| 名称 | 属性 | HP | 攻击 | 防御 | 闪避 |
|------|------|----|------|------|------|
| Pikachu | 电 | 80 | 35 | 5 | 30% |
| Bulbasaur | 草 | 100 | 35 | 10 | 10% |
| Squirtle | 水 | 80 | 25 | 20 | 20% |
| Charmander | 火 | 80 | 35 | 15 | 10% |
| Mew | 超能 | 90 | 40 | 10 | 20% |

## 属性克制

| 属性 | 克制(2x) | 被克制(0.5x) |
|------|---------|-------------|
| 草 | 水 | 火 |
| 火 | 草 | 水 |
| 水 | 火 | 电 |
| 电 | 水 | 超能 |
| 超能 | 电 | 草 |

## 属性被动

- **草**: 每回合回复10%HP
- **火**: 每次攻击叠加10%攻击力（最多4层）
- **水**: 受到伤害50%减免30%
- **电**: 闪避时立即反击
- **超能**: 30%概率每回合回复15%HP

In [1]:
from __future__ import annotations
import random

## 效果 (effects)

In [ ]:
class Effect:
    """效果基类"""
    name: str

    def __init__(self, duration: int) -> None:
        self.duration = duration

    def apply(self, pokemon: "Pokemon") -> None:
        raise NotImplementedError

    def decrease_duration(self) -> None:
        self.duration -= 1
        print(f"{self.name} 效果剩余回合: {self.duration}")


class PoisonEffect(Effect):
    """中毒：每回合损失10%最大HP"""
    name = "Poison"

    def __init__(self, duration: int = 3) -> None:
        super().__init__(duration)

    def apply(self, pokemon: "Pokemon") -> None:
        damage = max(int(pokemon.max_hp * 0.1), 1)
        pokemon.receive_damage(damage)
        print(f"{pokemon.name} 受到 {damage} 点中毒伤害！")


class BurnEffect(Effect):
    """烧伤：每回合固定10点伤害"""
    name = "Burn"

    def __init__(self, damage: int = 10, duration: int = 2) -> None:
        super().__init__(duration)
        self.damage = damage

    def apply(self, pokemon: "Pokemon") -> None:
        pokemon.receive_damage(self.damage)
        print(f"{pokemon.name} 受到 {self.damage} 点烧伤伤害！")


class HealEffect(Effect):
    """回复：每回合回血"""
    name = "Heal"

    def __init__(self, amount: int, duration: int = 3) -> None:
        super().__init__(duration)
        self.amount = amount

    def apply(self, pokemon: "Pokemon") -> None:
        pokemon.heal_self(self.amount)
        print(f"{pokemon.name} 回复了 {self.amount} 点HP！")

## 技能 (skills)

In [3]:
class Skill:
    """技能基类"""
    name: str

    def __init__(self, multiplier: float = 0) -> None:
        self.multiplier = multiplier
        self.needs_charge = False

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        raise NotImplementedError

    def calculate_damage(self, user: "Pokemon", opponent: "Pokemon") -> int:
        base = int(user.attack * self.multiplier)
        effectiveness = user.type_effectiveness(opponent)
        total = int(base * effectiveness)

        if effectiveness == 2.0:
            print("效果拔群！")
        elif effectiveness == 0.5:
            print("效果不太好...")

        if user.type == "Fire" and self.multiplier > 0:
            boost = 1 + 0.1 * user.attack_boost
            total = int(total * boost)
            user.attack_boost = min(user.attack_boost + 1, 4)
            print(f"[火属性] {user.name} 的攻击力提升了！({user.attack_boost}/4)")

        return total


class Thunderbolt(Skill):
    """十万伏特：1.4倍电属性伤害，10%麻痹"""
    name = "十万伏特"

    def __init__(self) -> None:
        super().__init__(multiplier=1.4)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        total = self.calculate_damage(user, opponent)
        opponent.receive_damage(total)
        if random.random() < 0.1:
            opponent.paralyzed = True
            print(f"{opponent.name} 麻痹了！")


class QuickAttack(Skill):
    """电光一闪：1.0倍攻击，10%触发第二次"""
    name = "电光一闪"

    def __init__(self) -> None:
        super().__init__(multiplier=1.0)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        total = self.calculate_damage(user, opponent)
        opponent.receive_damage(total)
        if random.random() < 0.1:
            print(f"{user.name} 的电光一闪再次攻击！")
            total2 = self.calculate_damage(user, opponent)
            opponent.receive_damage(total2)


class SeedBomb(Skill):
    """种子炸弹：草属性伤害，15%中毒"""
    name = "种子炸弹"

    def __init__(self) -> None:
        super().__init__(multiplier=1.2)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        total = self.calculate_damage(user, opponent)
        opponent.receive_damage(total)
        if random.random() < 0.15:
            opponent.add_status_effect(PoisonEffect(duration=3))
            print(f"{opponent.name} 中毒了！")


class ParasiticSeeds(Skill):
    """寄生种子：种下种子，每回合吸血"""
    name = "寄生种子"

    def __init__(self) -> None:
        super().__init__(multiplier=0)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        if opponent.leech_seed_turns > 0:
            print("已经种过寄生种子了！")
            return
        opponent.leech_seed_turns = 3
        opponent.leech_seed_source = user
        print(f"{opponent.name} 被种下了寄生种子！")


class AquaJet(Skill):
    """水枪：1.4倍水属性伤害"""
    name = "水枪"

    def __init__(self) -> None:
        super().__init__(multiplier=1.4)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        total = self.calculate_damage(user, opponent)
        opponent.receive_damage(total)


class Shield(Skill):
    """护盾：不造成伤害，下回合减伤50%"""
    name = "护盾"

    def __init__(self) -> None:
        super().__init__(multiplier=0)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        user.shield = True
        print(f"{user.name} 使用了护盾！下次伤害减半！")


class Ember(Skill):
    """火花：1.0倍火属性伤害，10%烧伤"""
    name = "火花"

    def __init__(self) -> None:
        super().__init__(multiplier=1.0)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        total = self.calculate_damage(user, opponent)
        opponent.receive_damage(total)
        if random.random() < 0.1:
            opponent.add_status_effect(BurnEffect(damage=10, duration=2))
            print(f"{opponent.name} 烧伤了！")


class FlameCharge(Skill):
    """蓄能爆炎：3.0倍火属性伤害，需蓄力，80%烧伤"""
    name = "蓄能爆炎"

    def __init__(self) -> None:
        super().__init__(multiplier=3.0)
        self.needs_charge = True
        self.raise_enemy_evasion = True

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        total = self.calculate_damage(user, opponent)
        opponent.receive_damage(total)
        if random.random() < 0.8:
            opponent.add_status_effect(BurnEffect(damage=10, duration=2))
            print(f"{opponent.name} 被严重烧伤了！")


class PsyShock(Skill):
    """精神冲击：1.3倍超能伤害，20%混乱"""
    name = "精神冲击"

    def __init__(self) -> None:
        super().__init__(multiplier=1.3)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        total = self.calculate_damage(user, opponent)
        opponent.receive_damage(total)
        if random.random() < 0.2 and opponent.confused_turns <= 0:
            opponent.confused_turns = 2
            print(f"{opponent.name} 混乱了！")


class Recover(Skill):
    """自我再生：回复30%HP"""
    name = "自我再生"

    def __init__(self) -> None:
        super().__init__(multiplier=0)

    def execute(self, user: "Pokemon", opponent: "Pokemon") -> None:
        heal = int(user.max_hp * 0.3)
        user.heal_self(heal)
        print(f"{user.name} 回复了 {heal} 点HP！({user.hp}/{user.max_hp})")

## 宝可梦 (pokemon)

In [ ]:
class Pokemon:
    """宝可梦基类"""
    name: str
    type: str

    def __init__(self, hp: int, attack: int, defense: int) -> None:
        self.hp = hp
        self.max_hp = hp
        self.attack = attack
        self.defense = defense
        self.skills = self.initialize_skills()
        self.alive = True
        self.statuses = []
        self.evasion = 0.0
        self.paralyzed = False
        self.confused_turns = 0
        self.leech_seed_turns = 0
        self.leech_seed_source = None
        self.shield = False 
        self.charging = False 
        self.charged_skill = None
        self.attack_boost = 0

    def initialize_skills(self):
        raise NotImplementedError

    def use_skill(self, skill: Skill, opponent: Pokemon) -> None:
        print(f"{self.name} 使用了 {skill.name}！")
        if not self.can_act():
            return
        if self.check_dodge(opponent):
            return
        skill.execute(self, opponent)

    def can_act(self) -> bool:
        if self.paralyzed and random.random() < 0.25:
            print(f"{self.name} 麻痹了，无法行动！")
            return False
        if self.confused_turns > 0:
            self.confused_turns -= 1
            if random.random() < 0.3:
                dmg = max(int(self.attack * 0.3), 1)
                self.hp -= dmg
                print(f"{self.name} 混乱了，打了自己 {dmg} 点伤害！")
                if self.hp <= 0:
                    self.alive = False
                    print(f"{self.name} 倒下了！")
                return False
            if self.confused_turns <= 0:
                print(f"{self.name} 恢复了清醒！")
        return True

    def check_dodge(self, attacker: Pokemon) -> bool:
        if random.random() < self.evasion:
            print(f"{attacker.name} 的攻击被躲开了！{self.name} 灵巧地闪避了！")
            if self.type == "Electric" and self.alive:
                dmg_skills = [s for s in self.skills if s.multiplier > 0]
                if dmg_skills:
                    cs = random.choice(dmg_skills)
                    print(f"[电属性] {self.name} 发动反击！")
                    cs.execute(self, attacker)
            return True
        return False

    def receive_damage(self, damage: int) -> None:
        damage = max(damage - self.defense, 1)
        if self.type == "Water" and random.random() < 0.5:
            damage = int(damage * 0.7)
            print(f"[水属性] {self.name} 的水之柔体减免了伤害！")
        if self.shield:
            damage = int(damage * 0.5)
            self.shield = False
            print(f"{self.name} 的护盾吸收了部分伤害！")
        self.hp -= damage
        print(f"{self.name} 受到 {damage} 点伤害！剩余HP：{self.hp}/{self.max_hp}")
        if self.hp <= 0:
            self.alive = False
            print(f"{self.name} 倒下了！")

    def heal_self(self, amount: int) -> None:
        if not isinstance(amount, int):
            amount = int(amount)
        self.hp = min(self.hp + amount, self.max_hp)

    def add_status_effect(self, effect: Effect) -> None:
        self.statuses.append(effect)

    def apply_status_effect(self) -> None:
        for s in self.statuses[:]:
            s.apply(self)
            s.decrease_duration()
            if s.duration <= 0:
                print(f"{self.name} 的 {s.name} 效果解除了。")
                self.statuses.remove(s)

    def begin(self) -> None:
        self.apply_status_effect()
        if not self.alive:
            return
        if self.leech_seed_turns > 0 and self.leech_seed_source and self.leech_seed_source.alive:
            dmg = max(int(self.max_hp * 0.1), 1)
            self.hp -= dmg
            self.leech_seed_source.heal_self(dmg)
            print(f"[寄生种子] {self.name} 被吸取了 {dmg} 点HP！")
            self.leech_seed_turns -= 1
            if self.leech_seed_turns <= 0:
                self.leech_seed_source = None
            if self.hp <= 0:
                self.alive = False
                print(f"{self.name} 倒下了！")

    def type_effectiveness(self, opponent: Pokemon) -> float:
        raise NotImplementedError

    def __str__(self) -> str:
        return f"{self.name}（{self.type}属性）HP:{self.hp}/{self.max_hp}"


class GrassPokemon(Pokemon):
    type = "Grass"

    def type_effectiveness(self, opponent: Pokemon) -> float:
        effectiveness = 1.0
        if opponent.type == "Water":
            effectiveness = 2.0
        elif opponent.type == "Fire":
            effectiveness = 0.5
        return effectiveness

    def begin(self) -> None:
        super().begin()
        if not self.alive:
            return
        heal = max(int(self.max_hp * 0.1), 1)
        if self.hp < self.max_hp:
            self.heal_self(heal)
            print(f"[草属性] {self.name} 进行光合作用，回复了 {heal} 点HP！({self.hp}/{self.max_hp})")


class FirePokemon(Pokemon):
    type = "Fire"

    def type_effectiveness(self, opponent: Pokemon) -> float:
        effectiveness = 1.0
        if opponent.type == "Grass":
            effectiveness = 2.0
        elif opponent.type == "Water":
            effectiveness = 0.5
        return effectiveness


class WaterPokemon(Pokemon):
    type = "Water"

    def type_effectiveness(self, opponent: Pokemon) -> float:
        effectiveness = 1.0
        if opponent.type == "Fire":
            effectiveness = 2.0
        elif opponent.type == "Electric":
            effectiveness = 0.5
        return effectiveness


class ElectricPokemon(Pokemon):
    type = "Electric"

    def type_effectiveness(self, opponent: Pokemon) -> float:
        effectiveness = 1.0
        if opponent.type == "Water":
            effectiveness = 2.0
        elif opponent.type == "Psychic":
            effectiveness = 0.5
        return effectiveness


class PsychicPokemon(Pokemon):
    type = "Psychic"

    def type_effectiveness(self, opponent: Pokemon) -> float:
        effectiveness = 1.0
        if opponent.type == "Electric":
            effectiveness = 2.0
        elif opponent.type == "Grass":
            effectiveness = 0.5
        return effectiveness

    def begin(self) -> None:
        super().begin()
        if not self.alive:
            return
        if random.random() < 0.3:
            heal = max(int(self.max_hp * 0.15), 1)
            self.heal_self(heal)
            print(f"[超能属性] {self.name} 的超能力回复了 {heal} 点HP！({self.hp}/{self.max_hp})")


class Pikachu(ElectricPokemon):
    name = "皮卡丘"

    def __init__(self, hp: int = 80, attack: int = 35, defense: int = 5) -> None:
        super().__init__(hp, attack, defense)
        self.evasion = 0.30

    def initialize_skills(self) -> list:
        return [Thunderbolt(), QuickAttack()]


class Bulbasaur(GrassPokemon):
    name = "妙蛙种子"

    def __init__(self, hp: int = 100, attack: int = 35, defense: int = 10) -> None:
        super().__init__(hp, attack, defense)
        self.evasion = 0.10

    def initialize_skills(self) -> list:
        return [SeedBomb(), ParasiticSeeds()]


class Squirtle(WaterPokemon):
    name = "杰尼龟"

    def __init__(self, hp: int = 80, attack: int = 25, defense: int = 20) -> None:
        super().__init__(hp, attack, defense)
        self.evasion = 0.20

    def initialize_skills(self) -> list:
        return [AquaJet(), Shield()]


class Charmander(FirePokemon):
    name = "小火龙"

    def __init__(self, hp: int = 80, attack: int = 35, defense: int = 15) -> None:
        super().__init__(hp, attack, defense)
        self.evasion = 0.10

    def initialize_skills(self) -> list:
        return [Ember(), FlameCharge()]


class Mew(PsychicPokemon):
    name = "梦幻"

    def __init__(self, hp: int = 90, attack: int = 40, defense: int = 10) -> None:
        super().__init__(hp, attack, defense)
        self.evasion = 0.20

    def initialize_skills(self) -> list:
        return [PsyShock(), Recover()]


ALL_POKEMON = [Pikachu, Bulbasaur, Squirtle, Charmander, Mew]

## 游戏流程 (play)

In [5]:
def valid_choice(choice: str, max_val: int) -> bool:
    return choice.isdigit() and 1 <= int(choice) <= max_val


class Play:
    """游戏主控类"""

    def __init__(self) -> None:
        self.player_team: list[Pokemon] = []
        self.computer_team: list[Pokemon] = []
        self.current_player: Pokemon | None = None
        self.current_computer: Pokemon | None = None
        self.turn: int = 0

    def print_list(self, pokemon_list: list[Pokemon]) -> None:
        for i, p in enumerate(pokemon_list, 1):
            status = "" if p.alive else " [已昏厥]"
            print(f"{i}: {p}{status}")

    def player_choose_team(self) -> None:
        available = list(ALL_POKEMON)
        print("\n请选择3只宝可梦组成队伍（输入编号，用空格分隔）：")
        for i, cls in enumerate(available, 1):
            print(f"{i}: {cls.name}（{cls.type}属性）")

        while True:
            choice = input("选择3只(如 1 2 3): ")
            nums = choice.split()
            if len(nums) != 3:
                print("请输入3个编号！")
                continue
            if not all(valid_choice(n, len(available)) for n in nums):
                print("请输入有效的编号(1-5)！")
                continue
            if len(set(nums)) != 3:
                print("不能选择重复的宝可梦！")
                continue
            for n in nums:
                cls = available[int(n) - 1]
                new_p = cls()
                self.player_team.append(new_p)
                print(f"{new_p.name} 加入了队伍！")
            break

        print(f"\n你的队伍：{'、'.join(p.name for p in self.player_team)}")

    def computer_choose_team(self) -> None:
        chosen = random.sample(ALL_POKEMON, 3)
        for cls in chosen:
            self.computer_team.append(cls())
        print(f"电脑的队伍：{', '.join(p.name for p in self.computer_team)}")

    def player_choose_active(self) -> Pokemon:
        alive = [p for p in self.player_team if p.alive]
        print("\n选择出战的宝可梦：")
        self.print_list(alive)

        while True:
            choice = input("输入编号：")
            if valid_choice(choice, len(alive)):
                p = alive[int(choice) - 1]
                self.current_player = p
                print(f"上吧，{p.name}！")
                return p
            print("无效选择！")

    def computer_choose_active(self) -> Pokemon:
        alive = [p for p in self.computer_team if p.alive]
        p = random.choice(alive)
        self.current_computer = p
        print(f"电脑派出了 {p.name}！")
        return p

    def player_use_skill(self) -> None:
        p = self.current_player

        if p.charging:
            skill = p.charged_skill
            p.charging = False
            p.charged_skill = None
            print(f"\n{p.name} 蓄力完成！")
            if getattr(skill, 'raise_enemy_evasion', False):
                self.current_computer.evasion += 0.2
            p.use_skill(skill, self.current_computer)
            if getattr(skill, 'raise_enemy_evasion', False):
                self.current_computer.evasion -= 0.2
            return

        print(f"\n为 {p.name} 选择技能：")
        for i, s in enumerate(p.skills, 1):
            tag = " [需蓄力]" if s.needs_charge else ""
            print(f"{i}: {s.name}{tag}")

        while True:
            choice = input("输入编号：")
            if valid_choice(choice, len(p.skills)):
                skill = p.skills[int(choice) - 1]
                if skill.needs_charge:
                    p.charging = True
                    p.charged_skill = skill
                    print(f"{p.name} 开始蓄力！")
                    return
                p.use_skill(skill, self.current_computer)
                return
            print("无效选择！")

    def computer_use_skill(self) -> None:
        p = self.current_computer

        if p.charging:
            skill = p.charged_skill
            p.charging = False
            p.charged_skill = None
            print(f"\n{p.name} 蓄力完成！")
            if getattr(skill, 'raise_enemy_evasion', False):
                self.current_player.evasion += 0.2
            p.use_skill(skill, self.current_player)
            if getattr(skill, 'raise_enemy_evasion', False):
                self.current_player.evasion -= 0.2
            return

        skill = random.choice(p.skills)
        if skill.needs_charge:
            p.charging = True
            p.charged_skill = skill
            print(f"{p.name} 开始蓄力！")
            return
        p.use_skill(skill, self.current_player)

    def check_status(self) -> str | None:
        player_alive = [p for p in self.player_team if p.alive]
        comp_alive = [p for p in self.computer_team if p.alive]

        if not player_alive and not comp_alive:
            print("\n双方全部倒下了！平局！")
            return "tie"
        if not player_alive:
            print("\n你的宝可梦全部倒下了！你输了...")
            return "lose"
        if not comp_alive:
            print("\n电脑的宝可梦全部倒下了！你赢了！")
            return "win"

        if self.current_player and not self.current_player.alive:
            print(f"{self.current_player.name} 倒下了！")
            self.player_choose_active()
        if self.current_computer and not self.current_computer.alive:
            print(f"{self.current_computer.name} 倒下了！")
            self.computer_choose_active()

        return None

    def run(self) -> None:
        print("=" * 45)
        print("   欢迎来到宝可梦对战竞技场！")
        print("=" * 45)

        self.player_choose_team()
        self.computer_choose_team()

        print("\n--- 选择首发宝可梦 ---")
        self.current_player = self.player_choose_active()
        self.current_computer = self.computer_choose_active()

        while True:
            self.turn += 1
            print(f"\n{'='*40}")
            print(f"       === 第{self.turn}回合 ===")
            print(f"{'='*40}")
            print(f"{self.current_player.name} HP: {self.current_player.hp}/{self.current_player.max_hp}")
            print(f"{self.current_computer.name} HP: {self.current_computer.hp}/{self.current_computer.max_hp}")

            print("\n--- 回合开始效果 ---")
            self.current_player.begin()
            self.current_computer.begin()

            if self.check_status():
                break

            print("\n--- 你的回合 ---")
            self.player_use_skill()
            if self.check_status():
                break

            print("\n--- 电脑回合 ---")
            self.computer_use_skill()
            if self.check_status():
                break

## 开始游戏

In [6]:
game = Play()
game.run()

   欢迎来到宝可梦对战竞技场！

请选择3只宝可梦组成队伍（输入编号，用空格分隔）：
1: 皮卡丘（Electric属性）
2: 妙蛙种子（Grass属性）
3: 杰尼龟（Water属性）
4: 小火龙（Fire属性）
5: 梦幻（Psychic属性）
皮卡丘 加入了队伍！
杰尼龟 加入了队伍！
梦幻 加入了队伍！

你的队伍：皮卡丘、杰尼龟、梦幻
电脑的队伍：杰尼龟, 皮卡丘, 梦幻

--- 选择首发宝可梦 ---

选择出战的宝可梦：
1: 皮卡丘（Electric属性）HP:80/80
2: 杰尼龟（Water属性）HP:80/80
3: 梦幻（Psychic属性）HP:90/90
无效选择！
上吧，杰尼龟！
电脑派出了 杰尼龟！

       === 第1回合 ===
杰尼龟 HP: 80/80
杰尼龟 HP: 80/80

--- 回合开始效果 ---

--- 你的回合 ---

为 杰尼龟 选择技能：
1: 水枪
2: 护盾
无效选择！
杰尼龟 使用了 水枪！
[水属性] 杰尼龟 的水之柔体减免了伤害！
杰尼龟 受到 10 点伤害！剩余HP：70/80

--- 电脑回合 ---
杰尼龟 使用了 水枪！
杰尼龟 受到 15 点伤害！剩余HP：65/80

       === 第2回合 ===
杰尼龟 HP: 65/80
杰尼龟 HP: 70/80

--- 回合开始效果 ---

--- 你的回合 ---

为 杰尼龟 选择技能：
1: 水枪
2: 护盾
杰尼龟 使用了 水枪！
杰尼龟 的攻击被躲开了！杰尼龟 灵巧地闪避了！

--- 电脑回合 ---
杰尼龟 使用了 水枪！
杰尼龟 受到 15 点伤害！剩余HP：50/80

       === 第3回合 ===
杰尼龟 HP: 50/80
杰尼龟 HP: 70/80

--- 回合开始效果 ---

--- 你的回合 ---

为 杰尼龟 选择技能：
1: 水枪
2: 护盾
杰尼龟 使用了 水枪！
杰尼龟 的攻击被躲开了！杰尼龟 灵巧地闪避了！

--- 电脑回合 ---
杰尼龟 使用了 护盾！
杰尼龟 使用了护盾！下次伤害减半！

       === 第4回合 ===
杰尼龟 HP: 50/80
杰尼龟 HP: 70/80

--- 回合开始效果 ---

--- 你的回合 ---

